# Problem Setup
Input: 
- Medical image
- Natural Language question

Output:
- Short textual answer (classification)

In [3]:
import os
import re
import json
from collections import Counter
from PIL import Image
import torch
import torch.nn as nn 
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import Dataset

### **1. Data Preprocessing**

- Using ImageNet's mean and stddev:
    - [stackoverflow discussion](https://stackoverflow.com/questions/58151507/why-pytorch-officially-use-mean-0-485-0-456-0-406-and-std-0-229-0-224-0-2)
    - [PyTorch documentation](https://docs.pytorch.org/vision/stable/models.html)

In [4]:
# 1.1 Image transformation
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

In [5]:
# 1.2 Text tokenization
# Simple word-level tokenizer
def tokenize(sentence):
    sentence = sentence.lower()
    sentence = re.sub(r"[^a-z0-9]", "", sentence)
    return sentence.split()

# Build vocab
def build_vocab(questions, min_freq=1):
    counter = Counter()
    for q in questions:
        counter.update(tokenize(q))

    vocab = {"<pad>": 0, "<unk>": 1}
    for word, freq, in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)
    return vocab

# Encode question
def encode_question(question, vocab, max_len=20):
    tokens = tokenize(question)
    ids = [vocab.get(w, vocab["<unk>"]) for w in tokens]
    ids = ids[:max_len]
    return ids + [vocab["<pad>"]] * (max_len - len(ids))

In [ ]:
class VqaRadDataset(Dataset):
    def __init__(self, json_path, image_dir, vocab, answer_to_index, transform=None):
        with open(json_path) as f:
            self.data = json.load(f)
        self.image_dir = image_dir
        self.vocab = vocab
        self.answer_to_index = answer_to_index
        self.transform = transform

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        item = self.data[index]

        image = Image.open(os.path.join(self.image_dir, item['image_name'])).convert('RGB')
        if self.transform:
            image = self.transform(image)

        question = torch.tensor(
            encode_question(item['question'], self.vocab), dtype=torch.long
        )

        answer = torch.tensor(
            self.answer_to_index[item['answer']], dtype=torch.long
        )

        return image, question, answer